Neural Networks for sentiment analysis.ipynb

Author: Jorge Luis Rosas Trigueros

Last modification: 20 apr 26

https://tinyurl.com/4cc7tyvk

In [1]:
# Install datasets if not already available
!pip install datasets

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from datasets import load_dataset
from collections import Counter
import numpy as np

In [3]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print("Train data:",train_data)
print("train_data[0]:",train_data[0])

In [4]:
def tokenize(text):
    return text.lower().split()

In [5]:
vocab_size = 10000
max_length = 256

counter = Counter()

for example in train_data:
    counter.update(tokenize(example["text"]))

most_common = counter.most_common(vocab_size - 2)

word2idx = {"<PAD>": 0, "<UNK>": 1}
idx2word = {0: "<PAD>", 1: "<UNK>"}

for i, (word, _) in enumerate(most_common, start=2):
    word2idx[word] = i
    idx2word[i] = word

In [6]:
# Download GloVe embeddings
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
embedding_dim = 100

glove_path = "glove.6B.100d.txt"
glove_embeddings = {}

with open(glove_path, "r", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

In [8]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word2idx.items():
    if word in glove_embeddings:
        embedding_matrix[idx] = glove_embeddings[word]
    else:
        # Random small values for unknown words
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

In [9]:
def encode(text):
    tokens = tokenize(text)
    indices = [word2idx.get(tok, 1) for tok in tokens]
    return indices[:max_length] + [0] * max(0, max_length - len(indices))

In [10]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, split):
        self.data = split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(label, dtype=torch.float32)

        return x, y

In [11]:
batch_size = 512

train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [12]:
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(
            torch.from_numpy(embedding_matrix)
        )
        self.embedding.weight.requires_grad = False  # freeze GloVe

        self.fc1 = nn.Linear(embedding_dim, 16)
        self.fc2 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)      # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)          # Global average pooling
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

#    x starts as word indices
# then    becomes embeddings
# then    becomes sentence vectors
# then    becomes hidden activations
# finally becomes a probability

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentNet(
    vocab_size,
    embedding_dim,
    embedding_matrix
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

In [15]:
import numpy as np

model.eval()

# Counters for confusion matrix
TP = 0  # True Positives
FP = 0  # False Positives
TN = 0  # True Negatives
FN = 0  # False Negatives

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs).squeeze()
        predictions = (outputs >= 0.5).float()

        for pred, gold in zip(predictions, labels):
            if pred == 1 and gold == 1:
                TP += 1
            elif pred == 1 and gold == 0:
                FP += 1
            elif pred == 0 and gold == 0:
                TN += 1
            elif pred == 0 and gold == 1:
                FN += 1


In [16]:
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

In [17]:
print("Confusion Matrix")
print("----------------")
print(f"TP: {TP}  FP: {FP}")
print(f"FN: {FN}  TN: {TN}")

print("\nMetrics")
print("-------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

In [18]:
def wrap_text_to_length(text, max_length):
    words = text.split()
    lines = []
    current_line = []
    current_line_length = 0

    for word in words:
        # Check if adding the next word (plus a space) exceeds max_length
        # If current_line is empty, we don't add a space before the first word.
        if not current_line or current_line_length + len(word) + 1 <= max_length:
            if current_line:
                current_line.append(' ')
                current_line_length += 1 # for the space
            current_line.append(word)
            current_line_length += len(word)
        else:
            # Current word doesn't fit, start a new line
            lines.append(''.join(current_line))
            print(''.join(current_line))
            current_line = [word]
            current_line_length = len(word)

    # Add the last line if it's not empty
    if current_line:
        lines.append(''.join(current_line))
        print(''.join(current_line))

    return lines

In [19]:
model.eval()

with torch.no_grad():
    sample_text = test_data[4]["text"]
    sample_tensor = torch.tensor(
        encode(sample_text), dtype=torch.long
    ).unsqueeze(0).to(device)

    prediction = model(sample_tensor)

wrap_text_to_length(sample_text,80)
print("Predicted probability of positive sentiment:", prediction.item())

Modified Network: Add an Extra Hidden Layer

In [20]:
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(
            torch.from_numpy(embedding_matrix)
        )
        self.embedding.weight.requires_grad = False  # freeze GloVe

        # Hidden layers
        self.fc1 = nn.Linear(embedding_dim, 32)
        self.fc2 = nn.Linear(32, 16)

        # Output layer
        self.fc3 = nn.Linear(16, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)      # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)          # Global average pooling
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [21]:
# Reinitialize Training Components

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentNet(
    vocab_size,
    embedding_dim,
    embedding_matrix
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [22]:
# Train the Modified Network

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

In [23]:
model.eval()

TP = FP = TN = FN = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs).squeeze()
        predictions = (outputs >= 0.5).float()

        for pred, gold in zip(predictions, labels):
            if pred == 1 and gold == 1:
                TP += 1
            elif pred == 1 and gold == 0:
                FP += 1
            elif pred == 0 and gold == 0:
                TN += 1
            elif pred == 0 and gold == 1:
                FN += 1

In [24]:
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

In [25]:
print("Confusion Matrix")
print("----------------")
print(f"TP: {TP}  FP: {FP}")
print(f"FN: {FN}  TN: {TN}")

print("\nMetrics")
print("-------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")